In [9]:
import altair as alt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

# import data

In [10]:
marathon = pd.read_csv("../data/marathon.csv")
marathon

,age,bmi,female,footwear,group,injury,mf_d,mf_di,mf_ti,max,sprint,mf_s,time_hrs
0,35,23.592323,0,2,1,2,42195,4,10295,60.0,1,4.098592,2.859722
1,33,22.518295,0,2,2,2,42195,3,12292,50.0,0,3.432720,3.414444
2,38,25.560312,0,2,3,1,42195,4,10980,65.0,0,3.842896,3.050000
3,34,22.607931,0,2,1,1,42195,3,10694,88.0,1,3.945670,2.970556
4,39,24.974836,0,2,1,1,42195,2,13452,51.0,0,3.136708,3.736667
...,...,...,...,...,...,...,...,...,...,...,...,...,...
924,23,23.277760,1,2,2,1,42195,3,15660,18.0,0,2.694444,4.350000
925,30,24.489796,0,2,2,1,42195,2,16110,45.0,0,2.619181,4.475000
926,44,24.237617,0,2,3,1,42195,2,12289,63.0,1,3.433558,3.413611
927,34,21.249750,0,2,3,1,42195,3,12602,32.0,0,3.348278,3.500556


In [11]:
marathon_50 = marathon.sample(n= 50, random_state = 300)
marathon_50_plot = alt.Chart(marathon_50).mark_circle().encode(
    x = alt.X("max").title("maximum distance ran per week (in miles) ").scale(zero = False),
    y = alt.Y("time_hrs").title("Race time").scale(zero = False)
)
marathon_50_plot

alt.Chart(...)

In [12]:
# We want to predict the race time for someone who 
# ran a maximum distance of 100 miles per week during training.
# With KNN Regression(k = 4)
marathon_100mile_time_prediction = (
    marathon_50.assign(diff = (100 - marathon_50["max"]).abs())
    .nsmallest(4,"diff")["time_hrs"]
    .mean()
)
marathon_100mile_time_prediction

np.float64(2.7840972222222224)

In [13]:
# Find the best k for estimation
# Cross Validation

# split the data
marathon_training, marathon_testing = train_test_split(
    marathon,
    test_size=0.25,
    random_state = 2000
)
# Predictors and Target
X_train = marathon_training[["max"]]  #dataframe [[]]
y_train = marathon_training["time_hrs"]   #A series []
X_test = marathon_testing[["max"]]
y_test = marathon_testing["time_hrs"]

# Preprocessor(Scaler)
preprocessor = StandardScaler()
# Pipeline
marathon_pipe = make_pipeline(preprocessor,KNeighborsRegressor())

# Cross Validation Results
marathon_cv = pd.DataFrame(
    cross_validate(
        marathon_pipe,
        X_train,
        y_train,
        cv = 5,
        scoring="neg_root_mean_squared_error",
        return_train_score = True
    )
)
marathon_cv

#Find the best k(GridSearchCV)
np.random.seed(2019)
param_grid = {"kneighborsregressor__n_neighbors" :range(1,201,1)}
marathon_tuned = GridSearchCV(
    marathon_pipe, param_grid, cv = 5, n_jobs= -1, 
    scoring="neg_root_mean_squared_error"
)
marathon_results = pd.DataFrame(
    marathon_tuned.fit(X_train, y_train).cv_results_
)
marathon_results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_kneighborsregressor__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.021588,0.011770,0.006424,0.001751,1,{'kneighborsregressor__n_neighbors': 1},-0.950554,-0.720655,-0.737264,-0.811824,-0.875788,-0.819217,0.085948,200
1,0.011616,0.009209,0.004924,0.001480,2,{'kneighborsregressor__n_neighbors': 2},-0.907774,-0.686601,-0.619281,-0.776516,-0.728222,-0.743679,0.096930,199
2,0.008223,0.000874,0.005074,0.000627,3,{'kneighborsregressor__n_neighbors': 3},-0.739712,-0.672443,-0.571857,-0.757362,-0.754195,-0.699114,0.070683,198
3,0.004541,0.002451,0.004800,0.005124,4,{'kneighborsregressor__n_neighbors': 4},-0.623204,-0.658248,-0.497471,-0.722489,-0.704339,-0.641150,0.079834,197
4,0.005867,0.002206,0.004654,0.002316,5,{'kneighborsregressor__n_neighbors': 5},-0.608111,-0.641772,-0.473009,-0.714606,-0.687010,-0.624901,0.084312,196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,0.004630,0.000907,0.009992,0.003366,196,{'kneighborsregressor__n_neighbors': 196},-0.563313,-0.623115,-0.441225,-0.640078,-0.621259,-0.577798,0.073053,170
196,0.004792,0.001884,0.006735,0.001343,197,{'kneighborsregressor__n_neighbors': 197},-0.563940,-0.623286,-0.442106,-0.640626,-0.621244,-0.578240,0.072812,173
197,0.005296,0.001465,0.008648,0.002366,198,{'kneighborsregressor__n_neighbors': 198},-0.563952,-0.623780,-0.442397,-0.640842,-0.621467,-0.578487,0.072827,175
198,0.004678,0.001270,0.006410,0.000585,199,{'kneighborsregressor__n_neighbors': 199},-0.564497,-0.624093,-0.442593,-0.641249,-0.621130,-0.578712,0.072801,179


In [14]:
marathon_min = marathon_tuned.best_params_
marathon_best_RMSPE = -marathon_tuned.best_score_

print(marathon_min, "\n ",marathon_best_RMSPE)

{'kneighborsregressor__n_neighbors': 107} 
  0.5669298834669159


In [15]:

# applying to Testing Data
np.random.seed(1234)
marathon_prediction = marathon_tuned.predict(X_test)
#get the error
marathon_rmspe = mean_squared_error(y_test, marathon_prediction)**0.5
marathon_rmspe

0.6116092762646357

In [16]:

#The error of train set is 0.5669298834669159
#The error of test set is 0.6116092762646357
#The model has worked pretty good

#Make a plot with trending line
np.random.seed(2019)
marathon_preds = marathon_training.assign(
    predictions = marathon_tuned.predict(X_train)
)
marathon_plot = (
    alt.Chart(marathon_preds).mark_circle(opacity = 0.4).encode(
        x = alt.X("max").title("the maximum distance run per week").scale(zero =False),
        y = alt.Y("time_hrs").title("marathon time").scale(zero = False)
    )
    +
    alt.Chart(marathon_preds).mark_line(color="black").encode(
        x = "max",
        y = "predictions"
    )
)
marathon_plot

alt.LayerChart(...)